# 🦀 Rust Interview MCQ — Part 8: Async, Futures & Tokio (Q701–Q800)


### Q701. What does the `Future` trait's `poll` method return?

- A) Option<Output>
- B) Result<Output, Error>
- C) Poll<Output>
- D) impl Future

<details>
<summary>Show Answer</summary>

**Q701. C)** Future::poll returns Poll<Output>, which is an enum: Ready(T) when complete, or Pending when not yet ready. The executor calls poll repeatedly until Ready.

</details>


### Q702. What is the role of `Context` in `Future::poll(&mut self, cx: &mut Context<'_>)`?

- A) It provides the future's output type
- B) It provides a Waker to notify the executor when the future can make progress
- C) It holds the future's state
- D) It is used for cancellation

<details>
<summary>Show Answer</summary>

**Q702. B)** Context wraps a Waker. When an async operation (e.g., I/O) completes, it calls waker.wake() so the executor knows to poll the future again. Without this, the executor would have to busy-poll.

</details>


### Q703. What does this async function desugar to?

```rust
async fn foo() -> i32 { 42 }
```

- A) A function that returns i32
- B) A function that returns impl Future<Output = i32>
- C) A function that returns Pin<Box<dyn Future<Output = i32>>>
- D) A synchronous function

<details>
<summary>Show Answer</summary>

**Q703. B)** async fn desugars to a regular function returning impl Future<Output = T>. The body becomes a future that, when polled, eventually yields T. The compiler generates a state machine.

</details>


### Q704. When you write `async { x.await }`, what is the type of the block?

- A) The type of x
- B) impl Future<Output = T> where T is the output of x
- C) Pin<Box<dyn Future>>
- D) Option<T>

<details>
<summary>Show Answer</summary>

**Q704. B)** An async block is an anonymous future. Its Output type is whatever the last await (or expression) produces. The block itself is impl Future<Output = T>.

</details>


### Q705. Why must `Future::poll` take `Pin<&mut Self>` rather than `&mut Self`?

- A) Pin is required for thread safety
- B) Futures may contain self-referential data; moving them could invalidate pointers
- C) Pin enables cancellation
- D) It's a historical artifact

<details>
<summary>Show Answer</summary>

**Q705. B)** Async/await generates state machines with self-referential structs (e.g., a future holding a reference into its own buffer). Moving such a value would invalidate the reference. Pin prevents moving, ensuring memory safety.

</details>


### Q706. What happens if a future returns `Poll::Pending` but never calls `waker.wake()`?

- A) The executor will panic
- B) The future will never be polled again; it effectively hangs
- C) The executor will poll it again after a fixed delay
- D) The future is automatically cancelled

<details>
<summary>Show Answer</summary>

**Q706. B)** Returning Pending without waking the waker means the executor has no way to know when to poll again. The task will never complete. Correct async code must call waker.wake() when progress becomes possible.

</details>


### Q707. What does `Waker::wake()` do?

- A) Immediately runs the associated task
- B) Schedules the task to be polled again by the executor
- C) Cancels the task
- D) Blocks until the task completes

<details>
<summary>Show Answer</summary>

**Q707. B)** wake() notifies the executor that the task is ready to make progress. The executor will (eventually) poll the future again. It does not run the task synchronously; it schedules it.

</details>


### Q708. In the desugared form of `async fn bar() { foo().await }`, what does `.await` become?

- A) A direct function call
- B) A loop that polls the future until Ready, passing the Context
- C) A thread spawn
- D) A yield point that returns immediately

<details>
<summary>Show Answer</summary>

**Q708. B)** .await desugars to a poll loop: repeatedly poll the future with the current Context until Poll::Ready. Each await point is a suspension point where the future can return Pending.

</details>


### Q709. Which is true about `Future` and `Send`?

- A) All futures are Send
- B) Futures that hold non-Send types across await points are not Send
- C) Futures are never Send because they use raw pointers
- D) Send is only required for multi-threaded executors

<details>
<summary>Show Answer</summary>

**Q709. B)** A future is Send only if all data held across await points is Send. If you hold an Rc (not Send) across an await, the future is !Send. Multi-threaded runtimes like Tokio require Send futures to move tasks between threads.

</details>


### Q710. What does `poll` returning `Poll::Ready` imply?

- A) The future will be polled again
- B) The future has produced its final output and will not be polled again
- C) The task is cancelled
- D) The Waker was invoked

<details>
<summary>Show Answer</summary>

**Q710. B)** Poll::Ready(output) means the future is complete. The executor consumes the output and will not poll this future again. The future's contract is that poll is not called again after Ready.

</details>


### Q711. What is the difference between `tokio::runtime::Runtime::new()` and `tokio::runtime::Builder::new_current_thread()`?

- A) They are identical
- B) new() creates a multi-threaded runtime; new_current_thread() uses a single thread
- C) new_current_thread() is faster but cannot spawn tasks
- D) new() is deprecated

<details>
<summary>Show Answer</summary>

**Q711. B)** Runtime::new() defaults to a multi-threaded work-stealing scheduler. new_current_thread() creates a single-threaded runtime—simpler, lower overhead, but no parallelism. Use current_thread for simple or embedded use cases.

</details>


### Q712. What does `tokio::spawn` return?

- A) The task's output
- B) JoinHandle<Output>
- C) impl Future
- D) ()

<details>
<summary>Show Answer</summary>

**Q712. B)** spawn returns JoinHandle<T> where T is the future's output type. You can await the handle to get the result, or drop it to detach the task. The handle also allows aborting the task.

</details>


### Q713. What happens if you drop a `JoinHandle` without awaiting it?

- A) The task is cancelled immediately
- B) The task continues running in the background (detached)
- C) The program panics
- D) The task is moved to the main thread

<details>
<summary>Show Answer</summary>

**Q713. B)** Dropping a JoinHandle detaches the task—it keeps running, but you can no longer retrieve its output. The task is not cancelled. Use JoinHandle::abort() to cancel.

</details>


### Q714. Can you call `tokio::spawn` outside of a Tokio runtime?

- A) Yes; it spawns a new runtime
- B) No; it panics if no runtime is in context
- C) Yes; it uses the global executor
- D) Only with #[tokio::main]

<details>
<summary>Show Answer</summary>

**Q714. B)** tokio::spawn requires a runtime to be current. If called outside (e.g., from a non-async main without #[tokio::main]), it panics. You must be inside a runtime or have one configured.

</details>


### Q715. What does `JoinHandle::abort()` do?

- A) Waits for the task to finish
- B) Cancels the task; the next poll will see a cancelled result
- C) Pauses the task
- D) Moves the task to another thread

<details>
<summary>Show Answer</summary>

**Q715. B)** abort() cancels the task. The task's future will receive a cancellation signal (typically via Drop or a cancelled result when polled). Awaiting the JoinHandle after abort yields JoinError::Cancelled.

</details>


### Q716. When using `#[tokio::main(flavor = "current_thread")]`, what happens if a spawned task blocks the thread?

- A) Other tasks run on a different thread
- B) All other tasks on that runtime are blocked until the blocking call completes
- C) The runtime spawns a new thread automatically
- D) Tokio detects blocking and yields

<details>
<summary>Show Answer</summary>

**Q716. B)** current_thread runtime has only one thread. If any task blocks (e.g., std::thread::sleep, CPU-bound work), no other tasks can run. Use tokio::task::spawn_blocking for blocking work.

</details>


### Q717. What is `tokio::task::spawn_blocking` for?

- A) Spawning tasks that run on a separate thread pool for blocking/sync code
- B) Spawning more async tasks
- C) Spawning tasks with higher priority
- D) Spawning tasks that cannot be cancelled

<details>
<summary>Show Answer</summary>

**Q717. A)** spawn_blocking runs synchronous/blocking code (e.g., file I/O, CPU work) on a dedicated thread pool so it doesn't block the async runtime. Returns a JoinHandle to a future that completes when the blocking work finishes.

</details>


### Q718. What does `JoinError::is_cancelled()` indicate?

- A) The task panicked
- B) The task was aborted via JoinHandle::abort()
- C) The runtime was shut down
- D) The task is still running

<details>
<summary>Show Answer</summary>

**Q718. B)** JoinError can represent cancellation (from abort()) or panic. is_cancelled() returns true when the task was explicitly aborted, as opposed to panicking.

</details>


### Q719. How do you get the output of a spawned task?

- A) The output is returned to the spawner automatically
- B) Await the JoinHandle: `handle.await`
- C) Use a channel passed to the task
- D) Both B and C work

<details>
<summary>Show Answer</summary>

**Q719. D)** You can await the JoinHandle to get Result<T, JoinError>. Alternatively, the task can send results through a channel. Awaiting the handle is the direct way; channels are useful for streaming or multiple results.

</details>


### Q720. What is `tokio::task::yield_now()` used for?

- A) To cancel the current task
- B) To voluntarily yield to the scheduler so other tasks can run
- C) To sleep for one tick
- D) To spawn a new task

<details>
<summary>Show Answer</summary>

**Q720. B)** yield_now() returns a future that completes on the next scheduler tick. It lets long-running CPU-bound async code cooperatively yield so other tasks get a chance to run. Prevents starving other tasks.

</details>


### Q721. Why does this fail to compile?

```rust
async fn foo(x: &i32) -> i32 { *x }
fn bar() -> impl Future<Output = i32> { foo(&42) }
```

- A) &42 does not live long enough
- B) The future returned by foo captures a reference; the future's lifetime is tied to x, but bar returns it without x
- C) async fn cannot take references
- D) 42 is not a valid reference

<details>
<summary>Show Answer</summary>

**Q721. B)** foo(&42) produces a future that holds a reference to a temporary. The future's Output = i32 is fine, but the future itself contains &i32. When bar returns, the temporary is gone. The future would outlive the reference—lifetime error.

</details>


### Q722. What does `async move` do in an async block?

- A) Moves the block to another thread
- B) Moves all captured variables into the future, transferring ownership
- C) Makes the future Send
- D) Moves the future to the heap

<details>
<summary>Show Answer</summary>

**Q722. B)** async move is like move on closures—it takes ownership of captured variables into the future. Without move, the future holds references; with move, it holds owned values. Use move when you need the future to own its data (e.g., for spawning).

</details>


### Q723. When does an `async fn` that takes `&self` produce a future that is not `Send`?

- A) Never; &self is always Send
- B) When the future holds &self across an await point and Self is not Send
- C) When the function is generic
- D) When it returns a reference

<details>
<summary>Show Answer</summary>

**Q723. B)** The generated future holds &self across await points. For the future to be Send, &self (and any other held data) must be Send. If Self contains Rc (not Send), &self across await makes the future !Send.

</details>


### Q724. What is the effect of `#[tokio::main(flavor = "multi_thread")]` on spawned task types?

- A) All spawned futures must be Send
- B) Futures need not be Send
- C) Only Sync types are allowed
- D) It has no effect on types

<details>
<summary>Show Answer</summary>

**Q724. A)** The multi-threaded runtime can move tasks between worker threads. Therefore, spawned futures must be Send so they can be safely moved. current_thread does not have this requirement.

</details>


### Q725. Why might `async fn get_string() -> &str` be problematic?

- A) async fn cannot return references
- B) It would require a named lifetime for the return type; the future would need to capture something with that lifetime
- C) &str is not Send
- D) It works fine

<details>
<summary>Show Answer</summary>

**Q725. B)** Returning &str from async fn means the future must produce a reference. That reference must outlive the future. Typically you'd need something like async fn get_string<'a>(s: &'a str) -> &'a str—the future holds &'a str. Complex lifetime constraints often lead to using owned types (String) instead.

</details>


### Q726. What does `Send` bound on a future ensure?

- A) The future can be cloned
- B) The future can be safely transferred between threads
- C) The future can be shared via Arc
- D) The future is thread-safe for concurrent polling

<details>
<summary>Show Answer</summary>

**Q726. B)** Send means the value can be transferred to another thread. For futures, this allows a multi-threaded runtime to move tasks between worker threads. The future must not hold non-Send data (e.g., Rc, raw pointers) across await points.

</details>


### Q727. How do you fix a future that is not Send due to holding Rc across await?

- A) Use Arc instead of Rc
- B) Add #[derive(Send)]
- C) Use Box::pin
- D) Wrap in Mutex

<details>
<summary>Show Answer</summary>

**Q727. A)** Rc is !Send (not thread-safe). Arc is Send (and Sync). Replacing Rc with Arc allows the future to be Send, as long as the inner type is also Send. Mutex can help with interior mutability but doesn't fix Rc.

</details>


### Q728. What is the type of `async { 1 + 2 }.await`?

- A) i32
- B) impl Future<Output = i32>
- C) Poll<i32>
- D) ()

<details>
<summary>Show Answer</summary>

**Q728. A)** The async block produces a future. .await polls it until Ready. The output of the future is i32 (from 1+2). So the whole expression has type i32.

</details>


### Q729. Can you implement Future for a type that holds a raw pointer?

- A) No; raw pointers make it unsafe
- B) Yes; you must ensure the future is never moved after being pinned, and that the pointer remains valid
- C) Only if the pointer is null
- D) Raw pointers are not allowed in async

<details>
<summary>Show Answer</summary>

**Q729. B)** You can implement Future with raw pointers, but you must uphold Pin's invariants: once pinned, the value must not be moved. Self-referential structs (common in async) use raw pointers; the compiler-generated state machines do this. It requires careful unsafe code.

</details>


### Q730. What does `Unpin` mean for a future?

- A) The future can be unpinned
- B) The future is safe to move after being pinned (it has no self-references that would be invalidated)
- C) The future cannot be pinned
- D) The future is thread-safe

<details>
<summary>Show Answer</summary>

**Q730. B)** Unpin is an auto trait. Types that don't contain Pin<P> (or similar) are Unpin. For Unpin futures, Pin<&mut T> is essentially &mut T—moving is safe. Most types are Unpin; async/await generates !Unpin when there are self-references.

</details>


### Q731. What does `tokio::select!` do?

- A) Runs multiple branches in parallel and returns all results
- B) Runs multiple futures and returns when the first one completes
- C) Selects one future to run based on a condition
- D) Runs futures sequentially

<details>
<summary>Show Answer</summary>

**Q731. B)** select! runs several futures concurrently and completes when the first one does. The other branches are cancelled. It's like a race—first to finish wins. Useful for timeouts, cancellation, or choosing between multiple async operations.

</details>


### Q732. What does `tokio::join!` do?

- A) Runs futures one after another
- B) Runs all futures concurrently and waits for all to complete, returning a tuple of results
- C) Joins two futures into one
- D) Runs the first future that completes

<details>
<summary>Show Answer</summary>

**Q732. B)** join! runs all branches concurrently and waits for every one to finish. Returns a tuple of all outputs. Unlike select!, nothing is cancelled—all must complete.

</details>


### Q733. In `tokio::select!`, what happens to the non-completed branches?

- A) They are dropped and their tasks cancelled
- B) They continue running in the background
- C) They are paused and can be resumed
- D) They are joined

<details>
<summary>Show Answer</summary>

**Q733. A)** When one branch of select! completes, the others are dropped. Dropping an incomplete future typically cancels it (the future's Drop runs). So only one branch "wins"; the rest are cancelled.

</details>


### Q734. What is `biased` in `tokio::select!`?

- A) It biases toward the first branch
- B) Branches are polled in order; the first ready branch wins, avoiding random selection
- C) It makes the select deterministic
- D) Both B and C

<details>
<summary>Show Answer</summary>

**Q734. D)** Without biased, select! may poll branches in a random order (to prevent starvation). biased makes it poll in source order—the first branch that is ready wins. This gives predictable behavior and can avoid starvation of earlier branches.

</details>


### Q735. What is a potential pitfall when using `select!` with a loop?

- A) The loop never terminates
- B) Creating new futures in each iteration can cause issues if you don't properly handle the previous iteration's futures
- C) select! cannot be used in loops
- D) Only one branch can be in a loop

<details>
<summary>Show Answer</summary>

**Q735. B)** A common bug: using `loop { select! { ... } }` where a branch creates a new future (e.g., a new TCP connection). If the future is created inside the branch and not stored, you might be selecting on a fresh future each time, or have borrow/lifetime issues. Need to structure the loop carefully.

</details>


### Q736. In `select! { a = fut1 => ..., b = fut2 => ... }`, when both futures are ready, which branch runs?

- A) Both run
- B) The first one in source order (with biased) or randomly (without biased)
- C) Neither; it's a race condition
- D) The one that became ready first

<details>
<summary>Show Answer</summary>

**Q736. B)** When multiple branches are ready, biased select! picks the first in source order. Without biased, Tokio randomizes to prevent starvation. Only one branch runs; the other is dropped.

</details>


### Q737. What does `tokio::try_join!` do compared to `tokio::join!`?

- A) They are identical
- B) try_join! short-circuits on the first Err, returning that error
- C) try_join! only works with Result types
- D) try_join! joins Option types

<details>
<summary>Show Answer</summary>

**Q737. B)** try_join! is for futures that return Result. It runs them concurrently but returns Err immediately if any returns Err, cancelling the rest. If all Ok, returns Ok((a, b, ...)).

</details>


### Q738. Why might `select!` cause a branch to "starve"?

- A) Without biased, random polling can favor a branch that is always ready
- B) A branch that is always ready (e.g., an already-received channel) can repeatedly win before others get a chance
- C) Both A and B
- D) select! prevents starvation by design

<details>
<summary>Show Answer</summary>

**Q738. C)** If one branch is always ready (e.g., receiving from a channel that already has data), it can keep winning. Without biased, randomization helps. With biased, the first always-ready branch wins every time—could starve later branches. Structure your branches or use fairness mechanisms.

</details>


### Q739. What does `fut.or(timeout)` do conceptually in a select!?

- A) Runs fut, and if it times out, runs timeout
- B) Runs both; the first to complete wins; typically timeout is a sleep that fires after a duration
- C) Runs fut with a timeout parameter
- D) Runs timeout after fut completes

<details>
<summary>Show Answer</summary>

**Q739. B)** In select!, you'd have something like `result = fut => ...` and `_ = tokio::time::sleep(duration) => ...`. Whichever completes first wins. The sleep branch implements a timeout—if the sleep fires first, you handle the timeout case.

</details>


### Q740. Can you use `select!` with more than 2 branches?

- A) No; only 2
- B) Yes; any number of branches
- C) Up to 4
- D) Only with a macro

<details>
<summary>Show Answer</summary>

**Q740. B)** select! accepts any number of branches. It runs all of them concurrently and completes when the first one does. Common to have 3+ (e.g., request, timeout, cancellation signal).

</details>


### Q741. What is the difference between `tokio::sync::mpsc::channel(1)` and `channel(100)`?

- A) They are identical
- B) The number is the buffer size; 1 means backpressure after one message
- C) 100 is the number of senders
- D) 1 is unbounded, 100 is bounded

<details>
<summary>Show Answer</summary>

**Q741. B)** The argument is the channel capacity. With 1, the sender blocks (or returns an error for try_send) when the buffer is full—backpressure. With 100, up to 100 messages can be buffered before backpressure.

</details>


### Q742. When does `mpsc::Sender::send()` return a future that completes?

- A) Immediately
- B) When the message has been received by the receiver
- C) When the message has been buffered (or when there is space in the buffer)
- D) When the receiver is dropped

<details>
<summary>Show Answer</summary>

**Q742. C)** send().await completes when the message can be buffered (or passed to the receiver if unbounded). It does not wait for the receiver to process the message. Backpressure: if the buffer is full, send blocks until space is available.

</details>


### Q743. What is `oneshot` channel used for?

- A) Sending multiple messages
- B) Single producer, single consumer, one message only
- C) Broadcasting to many receivers
- D) Synchronous communication

<details>
<summary>Show Answer</summary>

**Q743. B)** oneshot is for a single message: one sender, one receiver. Common for request-response patterns (e.g., spawn a task, send a oneshot sender; the task sends the result back). After one send, the channel is done.

</details>


### Q744. What does `broadcast` channel provide?

- A) Multiple senders, one receiver
- B) Multiple senders, multiple receivers; each message goes to all receivers
- C) One sender, multiple receivers
- D) Unbounded buffering

<details>
<summary>Show Answer</summary>

**Q744. B)** broadcast allows many senders and many receivers. Each sent message is cloned and delivered to every receiver. Useful for pub/sub, event broadcasting. Receivers that lag may miss messages (they get RecvError::Lagged).

</details>


### Q745. What is the `watch` channel used for?

- A) Sending a stream of values
- B) Holding a single value that can be updated; receivers see the current value
- C) Watching file changes
- D) Multiple consumers of one message

<details>
<summary>Show Answer</summary>

**Q745. B)** watch holds one value. The sender can update it. Receivers get the current value and can wait for changes. When the value changes, receivers are notified. Useful for configuration, state that multiple tasks need to observe.

</details>


### Q746. What happens when all `Receiver`s for an mpsc channel are dropped?

- A) The channel is closed; senders' send() will fail
- B) Nothing; the channel continues
- C) The senders panic
- D) The channel becomes unbounded

<details>
<summary>Show Answer</summary>

**Q746. A)** When all receivers are dropped, the channel is closed. Senders' send().await will return an error (the channel is closed). This allows graceful shutdown—receivers drop, senders detect it.

</details>


### Q747. What is backpressure in the context of async channels?

- A) Pressure from the OS
- B) When the receiver is slow, the sender blocks (or gets an error) instead of unbounded buffering
- C) A type of channel
- D) When the sender is dropped

<details>
<summary>Show Answer</summary>

**Q747. B)** Backpressure means slow consumers slow down producers. With a bounded channel, when the buffer is full, send blocks. This prevents fast producers from overwhelming memory. Unbounded channels have no backpressure.

</details>


### Q748. How do you get the latest value from a `watch::Receiver` without waiting?

- A) receiver.recv().await
- B) receiver.borrow() or receiver.get()—receiver has a .borrow() that gives &T
- C) receiver.try_recv()
- D) watch::Receiver has a .get() or .borrow() method for the current value

<details>
<summary>Show Answer</summary>

**Q748. D)** watch::Receiver has .borrow() (or in some versions, direct access) to get the current value without awaiting. .recv().await waits for the next change. So you can read the latest value synchronously.

</details>


### Q749. What does `mpsc::Sender::try_send()` return?

- A) Result<(), Error>
- B) Option<()>
- C) Result<(), SendError<T>>—Err contains the message if it couldn't be sent
- D) bool

<details>
<summary>Show Answer</summary>

**Q749. C)** try_send returns Result<(), SendError<T>>. Ok(()) on success. Err(SendError(msg)) if the channel is full or closed—the message is returned so you don't lose it.

</details>


### Q750. When using `broadcast`, what does `RecvError::Lagged` mean?

- A) The channel is closed
- B) The receiver fell behind; it missed some messages and was reset to the latest
- C) The sender was dropped
- D) The buffer overflowed

<details>
<summary>Show Answer</summary>

**Q750. B)** In broadcast, if the receiver is slow and the buffer overwrites, the receiver gets Lagged. It means messages were skipped. The receiver is typically reset to the newest value. You can detect and handle this.

</details>


### Q751. What does the `Stream` trait provide compared to `Future`?

- A) Stream is the same as Future
- B) Stream yields multiple values over time (like async Iterator)
- C) Stream is faster
- D) Stream handles errors

<details>
<summary>Show Answer</summary>

**Q751. B)** Stream::poll_next returns Poll<Option<Item>>. It yields a sequence of values, one at a time. Like Iterator but async. Future yields one value; Stream yields many.

</details>


### Q752. What does `StreamExt::next()` return?

- A) Option<Self::Item>
- B) impl Future<Output = Option<Self::Item>>
- C) Self::Item
- D) Vec<Self::Item>

<details>
<summary>Show Answer</summary>

**Q752. B)** next() returns a future that, when awaited, produces Option<Item>. Some(item) for the next value, None when the stream is exhausted. It consumes one item from the stream.

</details>


### Q753. How do you iterate over a Stream in async code?

- A) for await item in stream
- B) while let Some(item) = stream.next().await
- C) stream.for_each(|item| ...).await
- D) Both B and C work

<details>
<summary>Show Answer</summary>

**Q753. D)** Rust doesn't have for await yet. You use while let Some(x) = stream.next().await { } or for_each from StreamExt. for_each applies a closure to each item.

</details>


### Q754. What does `buffered(n)` do on a stream of futures?

- A) Buffers n items from the stream
- B) Runs up to n futures from the stream concurrently, preserving order of results
- C) Buffers the stream output
- D) Runs n streams in parallel

<details>
<summary>Show Answer</summary>

**Q754. B)** buffered(n) is for Stream<Item = impl Future>. It polls up to n futures at a time. Results are yielded in the same order as the stream's items. Like a concurrent map but ordered.

</details>


### Q755. What does `buffer_unordered(n)` do compared to `buffered(n)`?

- A) They are identical
- B) buffer_unordered yields results as they complete, not in stream order
- C) buffer_unordered uses less memory
- D) buffer_unordered is for unordered streams

<details>
<summary>Show Answer</summary>

**Q755. B)** buffer_unordered also runs up to n futures concurrently, but yields results in completion order (whichever future finishes first). buffered preserves the order of the stream. Use buffer_unordered when order doesn't matter and you want faster completion.

</details>


### Q756. What does `for_each_concurrent(n, f)` do?

- A) Runs the stream sequentially
- B) Applies f to up to n items concurrently as they're produced
- C) Runs n streams
- D) Buffers n items

<details>
<summary>Show Answer</summary>

**Q756. B)** for_each_concurrent limits concurrency. It pulls items from the stream and applies the async closure f to up to n at a time. When one finishes, the next is started. Useful for limiting concurrent work (e.g., HTTP requests).

</details>


### Q757. What does `StreamExt::collect` produce?

- A) A future that collects all items into a collection
- B) A Vec immediately
- C) The first item only
- D) A stream of collections

<details>
<summary>Show Answer</summary>

**Q757. A)** collect() returns a future. When awaited, it consumes the stream and collects all items into a type that implements FromStream (e.g., Vec). Like Iterator::collect but async.

</details>


### Q758. How do you convert an `Iterator` into a `Stream`?

- A) iterator.into_stream()
- B) Use futures::stream::iter() to wrap the iterator
- C) Stream::from(iterator)
- D) iterator.stream()

<details>
<summary>Show Answer</summary>

**Q758. B)** futures::stream::iter(iterator) creates a Stream that yields the iterator's items. Each item is produced as a future that is immediately ready. Useful for mixing sync data with async pipelines.

</details>


### Q759. What does `StreamExt::filter` return?

- A) A filtered Vec
- B) A new Stream that yields only items matching the predicate
- C) Option<Item>
- D) A Future

<details>
<summary>Show Answer</summary>

**Q759. B)** filter takes a predicate and returns a Stream. Only items for which the predicate returns true are yielded. Lazy—like Iterator::filter. The predicate can be async (filter with async predicate exists in some APIs).

</details>


### Q760. What is `StreamExt::take(n)` for?

- A) Taking ownership
- B) Limiting the stream to at most n items
- C) Taking the nth item
- D) Taking items until a condition

<details>
<summary>Show Answer</summary>

**Q760. B)** take(n) limits the stream to the first n elements. After n items, the stream ends. Like Iterator::take. Useful for limiting how much you process.

</details>


### Q761. What does `AsyncReadExt::read_to_string` do?

- A) Reads a single character
- B) Reads until EOF and appends to a String
- C) Reads a fixed number of bytes
- D) Reads a line

<details>
<summary>Show Answer</summary>

**Q761. B)** read_to_string reads all bytes until EOF and appends them to the given String, as valid UTF-8. Returns the number of bytes read. Used for reading a whole file or stream into a string.

</details>


### Q762. What is the purpose of `tokio::io::BufReader`?

- A) To make reads faster by buffering
- B) To provide line-by-line reading (read_line) and reduce syscalls
- C) To compress data
- D) To encrypt data

<details>
<summary>Show Answer</summary>

**Q762. B)** BufReader wraps an AsyncRead and buffers input. This reduces syscalls (read in chunks) and enables methods like read_line that need to look ahead. Similar to std::io::BufReader but for async.

</details>


### Q763. What does `AsyncWriteExt::write_all` do?

- A) Writes one byte
- B) Writes the entire buffer, retrying until all bytes are written or an error occurs
- C) Writes to stdout
- D) Writes and flushes

<details>
<summary>Show Answer</summary>

**Q763. B)** write_all takes a &[u8] and writes until all bytes are written. It handles partial writes (retries). Returns an error if the write fails. Does not flush—use flush() separately if needed.

</details>


### Q764. When should you use `tokio::fs` instead of `std::fs`?

- A) Always; std::fs is deprecated
- B) When you need to perform file I/O without blocking the async runtime
- C) Only for network I/O
- D) std::fs is faster

<details>
<summary>Show Answer</summary>

**Q764. B)** std::fs blocks the thread. In an async context, blocking a worker thread stalls other tasks. tokio::fs uses blocking thread pools or async file APIs so the runtime stays responsive. Use tokio::fs in async code.

</details>


### Q765. What does `AsyncBufReadExt::read_line` return?

- A) String
- B) Result<String, Error>
- C) usize (number of bytes read, including newline)
- D) Option<String>

<details>
<summary>Show Answer</summary>

**Q765. C)** read_line appends to the given String and returns usize—the number of bytes read (including the newline). The String is passed by reference. Returns 0 on EOF. The method signature is read_line(&mut self, buf: &mut String) -> Result<usize>.

</details>


### Q766. What is `tokio::io::split` used for?

- A) Splitting a file
- B) Splitting a type implementing AsyncRead + AsyncWrite (e.g., TcpStream) into separate read and write halves for concurrent use
- C) Splitting a stream
- D) Splitting bytes

<details>
<summary>Show Answer</summary>

**Q766. B)** split takes a duplex type (Read + Write) and returns (ReadHalf, WriteHalf). Each half can be used independently, possibly in different tasks. Useful for full-duplex protocols where one task reads and another writes.

</details>


### Q767. Why might you wrap a `TcpStream` in `BufReader`?

- A) To encrypt the connection
- B) To use line-based or framed protocols (e.g., read_line, read_until) efficiently
- C) To make it faster for large transfers
- D) BufReader doesn't work with TcpStream

<details>
<summary>Show Answer</summary>

**Q767. B)** BufReader buffers input. For line-based protocols (HTTP headers, Redis, etc.), read_line and read_until need to peek ahead. BufReader provides that and reduces syscalls. For raw byte streaming, it may add latency.

</details>


### Q768. What does `AsyncWriteExt::flush` do?

- A) Clears the buffer
- B) Ensures all buffered data has been written to the underlying writer
- C) Closes the stream
- D) Resets the writer

<details>
<summary>Show Answer</summary>

**Q768. B)** flush() writes any buffered data to the underlying I/O. BufWriter and similar hold data in a buffer; flush ensures it's all sent. Important before closing or when you need data to be sent immediately.

</details>


### Q769. What is `tokio::io::copy` for?

- A) Copying files
- B) Copying bytes from an AsyncRead to an AsyncWrite (e.g., proxy data between streams)
- C) Cloning handles
- D) Copying buffers

<details>
<summary>Show Answer</summary>

**Q769. B)** copy(read, write) reads from the reader and writes to the writer until EOF. Returns the number of bytes copied. Useful for proxying (e.g., TCP relay), piping streams, or saving a download to a file.

</details>


### Q770. Can you use `std::net::TcpStream` with Tokio?

- A) Yes; it works directly
- B) No; you must use tokio::net::TcpStream
- C) Yes; wrap it with tokio::net::TcpStream::from_std
- D) Only for reading

<details>
<summary>Show Answer</summary>

**Q770. C)** tokio::net::TcpStream::from_std converts a std TcpStream to Tokio's. You might do this when integrating with sync code or when you obtain a std stream (e.g., from a library). The Tokio stream is non-blocking and works with the async runtime.

</details>


### Q771. What is the key difference between `tokio::sync::Mutex` and `std::sync::Mutex` in async code?

- A) They are identical
- B) tokio::sync::Mutex is async-aware; locking it yields to the runtime instead of blocking the thread
- C) std::sync::Mutex is faster
- D) tokio::sync::Mutex cannot deadlock

<details>
<summary>Show Answer</summary>

**Q771. B)** std::sync::Mutex::lock() blocks the thread. In async, blocking a worker thread stalls other tasks. tokio::sync::Mutex::lock() returns a future that yields when the lock is contended—other tasks can run. Use Tokio's Mutex in async code.

</details>


### Q772. When might you use `std::sync::Mutex` in async code?

- A) Never
- B) When the critical section is very short and you're sure it won't block (e.g., updating a counter)
- C) Always; it's faster
- D) Only with spawn_blocking

<details>
<summary>Show Answer</summary>

**Q772. B)** If the critical section is tiny (e.g., a few nanoseconds) and never does I/O or awaits, std::sync::Mutex can be acceptable—the lock is held so briefly that blocking is negligible. For longer holds or any await, use tokio::sync::Mutex.

</details>


### Q773. What is `tokio::sync::Semaphore` used for?

- A) Replacing Mutex
- B) Limiting concurrent access to a resource (e.g., max N tasks at a time)
- C) Signaling between tasks
- D) Lock-free programming

<details>
<summary>Show Answer</summary>

**Q773. B)** Semaphore has a permit count. acquire() takes a permit (waits if none available). release() returns one. Used to limit concurrency: e.g., max 10 concurrent HTTP requests. Also useful for rate limiting.

</details>


### Q774. What does `tokio::sync::Notify` provide?

- A) A mutex
- B) A one-shot or repeated wake-up signal; one or more tasks can wait, another can notify
- C) A channel
- D) A semaphore

<details>
<summary>Show Answer</summary>

**Q774. B)** Notify is a simple notification primitive. Tasks call notified().await to wait. Another task calls notify_one() or notify_waiters() to wake them. Useful for custom synchronization, e.g., "wake me when X is ready."

</details>


### Q775. What is `tokio::sync::RwLock` for?

- A) Replacing Mutex with a faster lock
- B) Allowing multiple readers or a single writer
- C) Read-only locks
- D) Write-only locks

<details>
<summary>Show Answer</summary>

**Q775. B)** RwLock allows many concurrent readers or one writer. read() for shared access, write() for exclusive. Better than Mutex when reads are common and writes rare. tokio::sync::RwLock is async-aware (yields when contended).

</details>


### Q776. Why can holding a `std::sync::MutexGuard` across an `.await` be problematic?

- A) The guard might be dropped
- B) It blocks the async worker thread; if another task needs the lock, you can deadlock
- C) The guard is not Send
- D) Both B and C

<details>
<summary>Show Answer</summary>

**Q776. D)** Holding a MutexGuard across await blocks the thread—no other task can run on that worker. If another task needs the same lock, deadlock. Also, MutexGuard may not be Send, so the future might not be Send (required for multi-threaded runtimes).

</details>


### Q777. What does `Semaphore::acquire_many(5)` do?

- A) Acquires 5 semaphores
- B) Acquires 5 permits from the semaphore in one call
- C) Acquires from 5 different semaphores
- D) Increases the permit count by 5

<details>
<summary>Show Answer</summary>

**Q777. B)** acquire_many(n) acquires n permits at once. Useful when a task needs multiple "slots" (e.g., a batch operation). The semaphore's available count decreases by n. Release the same number when done.

</details>


### Q778. How does `Notify` differ from a oneshot channel?

- A) Notify is faster
- B) Notify doesn't carry data; it's just a signal. Oneshot sends a value
- C) Notify can wake multiple waiters; oneshot is single consumer
- D) Both B and C

<details>
<summary>Show Answer</summary>

**Q778. D)** Notify is a bare signal—no data. Oneshot sends one value. Notify can notify_one or notify_waiters (wake all). Oneshot has one sender, one receiver, one value. Use Notify when you just need to wake tasks; use oneshot for request-response.

</details>


### Q779. What is `tokio::sync::MutexGuard` across await points?

- A) It is Send, so holding it across await is fine
- B) It is not Send; holding it across await can make the future !Send
- C) It automatically releases at await
- D) It causes a compile error

<details>
<summary>Show Answer</summary>

**Q779. B)** tokio::sync::MutexGuard is Send (unlike std's in some cases). But the key point: holding any lock across await extends the hold time. With tokio::Mutex, the future yields while waiting, so it's designed for this. The guard itself is Send. The main issue with std::Mutex is blocking + potential !Send.

</details>


### Q780. What is 'structured concurrency' in the context of async?

- A) Using structs for concurrency
- B) Ensuring child tasks are contained within a parent's lifetime; when the parent is cancelled, children are too
- C) A Tokio-specific feature
- D) Using select! only

<details>
<summary>Show Answer</summary>

**Q780. B)** Structured concurrency means child tasks don't outlive their parent. When the parent scope ends (or is cancelled), all children are cancelled. tokio::spawn doesn't enforce this—tasks can outlive the spawner. Scopes (e.g., tokio::task::scope in some runtimes) provide it.

</details>


### Q781. What is `tokio::task::LocalKey` (task-local storage) used for?

- A) Storing data in a global variable
- B) Storing data that is local to each task, like thread-local but per-task
- C) Storing data in a Mutex
- D) Storing data in a channel

<details>
<summary>Show Answer</summary>

**Q781. B)** Task-local storage holds data per task. Each task has its own copy. Useful for request IDs, tracing context, or other per-task state without passing it through every function. Similar to thread_local but for async tasks.

</details>


### Q782. Why is async Drop challenging in Rust?

- A) Drop is synchronous; you cannot await in drop
- B) Drop must complete synchronously; async cleanup (e.g., flushing, closing connections) cannot be awaited
- C) Drop cannot be implemented for async types
- D) Drop is automatically async

<details>
<summary>Show Answer</summary>

**Q782. B)** Drop::drop is sync and cannot be async. If a type needs async cleanup (flush a writer, close a connection gracefully), you can't do it in Drop. You need explicit async close methods (e.g., close().await) that callers must use before dropping.

</details>


### Q783. What does 'cancellation safety' mean for a future?

- A) The future cannot be cancelled
- B) If the future is dropped (cancelled) at an await point, no invariants are violated and resources are properly cleaned up
- C) The future uses a CancellationToken
- D) The future is in a select!

<details>
<summary>Show Answer</summary>

**Q783. B)** When a future is dropped (e.g., by select! when another branch wins), it may be at an await point. Cancellation-safe code ensures that dropping doesn't leave things in a bad state (e.g., half-written data, leaked resources). Some operations are not cancellation-safe.

</details>


### Q784. Which operation is typically NOT cancellation-safe?

- A) Reading from a channel
- B) Writing to a file with a series of write() calls—if cancelled mid-write, data may be partially written
- C) Sleeping
- D) Receiving from mpsc

<details>
<summary>Show Answer</summary>

**Q784. B)** A sequence of writes can leave partial data if cancelled between writes. Channel send/recv are usually atomic. Sleep is trivially safe (nothing to clean up). Multi-step I/O operations that must be atomic are problematic when cancelled.

</details>


### Q785. How can you make cleanup run when a task is cancelled?

- A) Implement Drop—it always runs
- B) Use a guard type with Drop; when the future is dropped, the guard's Drop runs
- C) Use CancellationToken::cancelled().await
- D) Both B and C are valid approaches

<details>
<summary>Show Answer</summary>

**Q785. D)** Drop runs when the future is dropped, so a guard struct with Drop can run cleanup. Alternatively, use a CancellationToken and have the task poll cancelled().await; when cancelled, run cleanup before exiting. Both work.

</details>


### Q786. What does `tokio::task::scope` (or similar) provide?

- A) A scope for variables
- B) Structured concurrency: spawned tasks are joined or cancelled when the scope exits
- C) A new runtime
- D) Task-local storage

<details>
<summary>Show Answer</summary>

**Q786. B)** Scoped spawning ensures child tasks don't outlive the scope. When the scope block ends, you either await all children or they're cancelled. Prevents orphaned tasks. Note: tokio's API has evolved; the exact name may vary.

</details>


### Q787. Why might you use `CancellationToken` instead of `JoinHandle::abort()`?

- A) CancellationToken is faster
- B) CancellationToken allows cooperative cancellation; the task can run cleanup when it observes cancellation
- C) abort() doesn't work
- D) CancellationToken works with select!

<details>
<summary>Show Answer</summary>

**Q787. B)** abort() forcefully drops the future—cleanup in Drop runs, but the task doesn't get to run arbitrary code. CancellationToken lets the task poll token.cancelled().await and, when triggered, run cleanup (flush, close) before exiting. Cooperative cancellation.

</details>


### Q788. What is a common pattern for async resource cleanup?

- A) Rely only on Drop
- B) Provide an explicit async close() or shutdown() method; document that callers should await it before dropping
- C) Use Arc::strong_count
- D) Never drop async types

<details>
<summary>Show Answer</summary>

**Q788. B)** Since Drop can't await, the pattern is: provide close().await (or similar). Callers must call it before dropping. Drop can log a warning if close wasn't called. Some types use a background task for cleanup.

</details>


### Q789. When a future in `select!` is dropped, what runs?

- A) Nothing
- B) The future's Drop implementation, if any
- C) A cancellation handler
- D) The runtime panics

<details>
<summary>Show Answer</summary>

**Q789. B)** When the losing branch's future is dropped, its Drop runs. Any guard types or resources with Drop get cleaned up. The future doesn't get to run more async code—Drop is sync. Design your futures so Drop leaves things in a safe state.

</details>


### Q790. What is connection pooling in async Rust?

- A) A pool of threads
- B) Reusing a set of connections (e.g., DB, HTTP) instead of creating new ones for each request
- C) A memory pool
- D) A channel

<details>
<summary>Show Answer</summary>

**Q790. B)** Connection pooling maintains a pool of pre-established connections. Requests check out a connection, use it, and return it. Avoids the overhead of connecting for every request. Common for databases, HTTP clients.

</details>


### Q791. What does exponential backoff mean for retries?

- A) Retry immediately
- B) Wait time doubles (or scales) after each failure: 1s, 2s, 4s, 8s...
- C) Retry at random intervals
- D) Retry a fixed number of times

<details>
<summary>Show Answer</summary>

**Q791. B)** Exponential backoff increases delay between retries: e.g., 1s, 2s, 4s, 8s. Reduces load on a failing service. Often combined with jitter (randomization) to avoid thundering herd.

</details>


### Q792. How do you add a timeout to an async operation in Tokio?

- A) tokio::time::timeout(duration, future).await
- B) future.timeout(duration)
- C) tokio::sleep(duration).await
- D) select! with sleep is the only way

<details>
<summary>Show Answer</summary>

**Q792. A)** tokio::time::timeout(duration, future) runs the future and returns Ok(result) if it completes in time, or Err(Elapsed) if it times out. The future is cancelled on timeout. Clean and idiomatic.

</details>


### Q793. What is graceful shutdown?

- A) Killing the process immediately
- B) Stopping acceptance of new work, finishing in-flight work, then exiting
- C) Restarting the server
- D) Closing all connections immediately

<details>
<summary>Show Answer</summary>

**Q793. B)** Graceful shutdown: stop accepting new connections/requests, let current work finish (or wait with a timeout), release resources, then exit. Ensures no in-flight requests are cut off. Often uses signals (SIGTERM) and cancellation tokens.

</details>


### Q794. How might you implement graceful shutdown with Tokio?

- A) Use tokio::signal::ctrl_c() to listen for SIGINT, then set a cancellation token or close a channel that tasks await
- B) Call std::process::exit
- C) Use spawn_blocking
- D) There is no way

<details>
<summary>Show Answer</summary>

**Q794. A)** Listen for ctrl_c() (or other signals). When received, set a CancellationToken or close a shutdown channel. Tasks that await this will wake up and exit. Optionally wait for them with a timeout, then exit.

</details>


### Q795. What does `tokio::time::interval` produce?

- A) A one-shot delay
- B) A stream that yields at fixed intervals (first tick immediate, then every period)
- C) A timeout
- D) A timer

<details>
<summary>Show Answer</summary>

**Q795. B)** interval(period) creates an Interval. tick().await yields immediately the first time, then every period. Used for periodic tasks (heartbeats, cleanup). Missed ticks can be skipped or batched depending on configuration.

</details>


### Q796. Why add jitter to retry backoff?

- A) To make it faster
- B) To avoid thundering herd—many clients retrying at the same time
- C) To reduce memory
- D) To improve security

<details>
<summary>Show Answer</summary>

**Q796. B)** Without jitter, many clients retry at the same time (e.g., all at 1s, 2s, 4s). This can overwhelm a recovering service. Jitter randomizes the delay so retries are spread out.

</details>


### Q797. What is a common pattern for a connection pool's `get()` method?

- A) Always create a new connection
- B) Try to get an idle connection from the pool; if none, create one (or wait); return the connection; on drop, return it to the pool
- C) Use a Mutex only
- D) Use a broadcast channel

<details>
<summary>Show Answer</summary>

**Q797. B)** get() checks for an idle connection. If available, return it. If not, create a new one (up to max) or wait. The returned connection often uses a guard that, when dropped, returns the connection to the pool. Semaphore can limit pool size.

</details>


### Q798. What does `tokio::join!` return when all branches complete successfully?

- A) Vec of results
- B) A tuple of the outputs, one per branch
- C) The first result
- D) ()

<details>
<summary>Show Answer</summary>

**Q798. B)** join!(a, b, c) returns (result_a, result_b, result_c) when all complete. Each result is the output of that future. Order matches the order of the branches.

</details>


### Q799. When using `timeout()` with a database query, what happens to the query on timeout?

- A) The query completes in the background
- B) The future is dropped; the query may still run on the server, but the client stops waiting
- C) The database cancels the query
- D) The connection is closed

<details>
<summary>Show Answer</summary>

**Q799. B)** timeout drops the future when it fires. The future's Drop runs. The underlying operation (e.g., TCP request) may still be in flight—the server might still execute the query. The client just stops waiting. True cancellation requires protocol support (e.g., cancel message).

</details>


### Q800. What does `tokio::time::Instant::now()` plus `Duration` give you when used with `sleep_until`?

- A) A future that sleeps until an absolute instant in time
- B) A relative delay
- C) An interval
- D) A timeout

<details>
<summary>Show Answer</summary>

**Q800. A)** sleep_until(instant) sleeps until that absolute Instant. Instant::now() + duration gives a future instant. Useful when you need to sleep until a specific wall-clock time (e.g., next midnight) rather than a relative duration. sleep(duration) is relative; sleep_until is absolute.

</details>
